<a href="https://colab.research.google.com/github/SuccessEmman/blv-project-eval/blob/main/analysis/cider_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pycocoevalcap

In [ ]:
import csv, collections, re
from pycocoevalcap.cider.cider import Cider

BASE = "/content/drive/MyDrive/BLV Project/Analysis/outputs"
REFS = f"{BASE}/captions_long.csv"
MODELS = {
    "blip2": f"{BASE}/generated_captions/blip2_final_results.csv",
    "llava": f"{BASE}/generated_captions/llava_final_results.csv",
}

def norm(s):
    s = (s or "").lower().replace("\n", " ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

refs = collections.defaultdict(list)
for r in csv.DictReader(open(REFS, encoding="utf-8")):
    c = norm(r["caption"])
    if c:
        refs[str(r["image_id"])].append(c)

cands, stratum = {}, {}
for name, path in MODELS.items():
    d = {}
    for r in csv.DictReader(open(path, encoding="utf-8")):
        iid = str(r["image_id"])
        d[iid] = [norm(r["caption"])]
        stratum[iid] = r.get("stratum", "")
    cands[name] = d

scorer, per_image_rows = Cider(), []
for name, cand in cands.items():
    ids = [i for i in cand if i in refs]
    missing = [i for i in cand if i not in refs]
    corpus, per = scorer.compute_score({i: refs[i] for i in ids}, {i: cand[i] for i in ids})
    print(f"{name.upper()}  scored {len(ids)}  (no reference: {len(missing)})  CORPUS CIDEr {corpus*100:.1f}")
    by = collections.defaultdict(list)
    for iid, s in zip(ids, per):
        by[stratum[iid]].append(s)
        per_image_rows.append({"model": name, "image_id": iid,
                               "stratum": stratum[iid], "cider": float(s)})
    for s in ["clear", "blurry", "low_light", "text"]:
        if by[s]:
            print(f"    {s:10s}: {sum(by[s])/len(by[s])*100:5.1f} (n={len(by[s])})")
    print()

with open(f"{BASE}/cider_per_image.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["model", "image_id", "stratum", "cider"])
    w.writeheader(); w.writerows(per_image_rows)
print(f"per-image CIDEr -> {BASE}/cider_per_image.csv  ({len(per_image_rows)} rows)")

BLIP2  scored 217  (no reference: 33)  CORPUS CIDEr 80.8
    clear     :  84.9 (n=70)
    blurry    :  66.5 (n=63)
    low_light :  33.5 (n=34)
    text      : 125.4 (n=50)

LLAVA  scored 217  (no reference: 33)  CORPUS CIDEr 0.1
    clear     :   0.1 (n=70)
    blurry    :   0.0 (n=63)
    low_light :   0.0 (n=34)
    text      :   0.5 (n=50)

per-image CIDEr -> /content/drive/MyDrive/BLV Project/Analysis/outputs/cider_per_image.csv  (434 rows)
